# Importar codigos

Se obtienen las funciones del codigo

In [12]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import subprocess

import import_ipynb
from Codigos import *
from Codigos_obj import calcular_ms_ssim_lpips

# Funcion para obtener todos los valores cuantitativos de la imagen

Obtiene los valores seleccionados de la parte teorica

In [22]:
def recodificar_imagen(img_path, metodo, formato, calidad):
    import os
    import subprocess
    from PIL import Image

    nombre = os.path.splitext(os.path.basename(img_path))[0]
    out_dir = f"imagenes/{metodo}/{formato}"
    os.makedirs(out_dir, exist_ok=True)

    out_path = f"{out_dir}/{nombre}_{metodo}_q{calidad}.{formato}"

    if metodo == "pillow":
        img = Image.open(img_path).convert("RGB")

        if formato == "jpg":
            img.save(out_path, "JPEG", quality=calidad)
        elif formato == "webp":
            img.save(out_path, "WEBP", quality=calidad)
        elif formato == "png":
            img.save(out_path, "PNG", compress_level=int((100 - calidad)/10))

    elif metodo == "ffmpeg":
        cmd = [
            "ffmpeg", "-y",
            "-i", img_path,
            "-q:v", str(int((100 - calidad)/10 + 2)),
            out_path
        ]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    elif metodo == "imagemagick":
        cmd = [
            "magick",
            img_path,
            "-quality", str(calidad),
            out_path
        ]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    else:
        raise ValueError("Metodo no soportado")

    return out_path

In [14]:
def features_histograma(path, prefix):
    import numpy as np

    f = {}

    hist, hist_norm = obtener_histograma_intensidades(path)

    # guardar todo
    f[f"{prefix}_hist"] = hist.tolist()
    f[f"{prefix}_hist_norm"] = hist_norm.tolist()
    f[f"{prefix}_hist_bins"] = list(range(256))

    # derivados
    hist_norm_safe = hist_norm[hist_norm > 0]

    f[f"{prefix}_hist_entropy"] = -np.sum(hist_norm_safe * np.log2(hist_norm_safe))
    f[f"{prefix}_hist_std"] = np.std(hist_norm)
    f[f"{prefix}_hist_mean"] = np.mean(hist_norm)
    f[f"{prefix}_hist_min"] = np.min(hist_norm)
    f[f"{prefix}_hist_max"] = np.max(hist_norm)

    mu = np.mean(hist_norm)
    f[f"{prefix}_hist_skew"] = np.mean((hist_norm - mu)**3)
    f[f"{prefix}_hist_kurt"] = np.mean((hist_norm - mu)**4)

    return f

In [15]:
def features_generales(path, prefix):
    f = {}

    # básicas
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    # contraste
    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)

    # frecuencia
    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    # bordes multi escala
    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    # gradiente
    f[f"{prefix}_grad_mean"] = obtener_magnitud_media_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)

    # textura
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_textura(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_textura(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    # color
    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)

    return f

In [16]:
def extraer_features(path, prefix):
    f = {}

    f.update(features_generales(path, prefix))
    f.update(features_histograma(path, prefix))

    return f

In [17]:
def procesar_imagen(
    img_path,
    df,
    metodo="pillow",
    formato="jpg",
    calidad=50
):
    import pandas as pd

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df debe ser un pandas.DataFrame")

    # 1. recodificar
    out_path = recodificar_imagen(img_path, metodo, formato, calidad)

    # 2. features
    feats_orig = extraer_features(img_path, "orig")
    feats_comp = extraer_features(out_path, "comp")

    # 3. métrica objetivo
    metricas = calcular_ms_ssim_lpips(img_path, out_path)

    # 4. fila
    fila = {
        "img_original": img_path,
        "img_recodificada": out_path,
        "metodo": metodo,
        "formato": formato,
        "calidad": calidad,
        **feats_orig,
        **feats_comp,
        **metricas
    }

    return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)

In [26]:
CALIDADES       = [10, 30, 50, 70, 90]
METODOS         = ["pillow", "ffmpeg", "imagemagick"]
FORMATOS        = ["jpg", "webp", "png"]
CARPETA_RAIZ    = "imagenes"

def procesar_dataset_completo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Itera sobre imagenes/jpg, imagenes/webp, imagenes/png.
    Por cada imagen aplica los 3 métodos × formatos válidos × 5 calidades.
    Nunca convierte una imagen al mismo formato que ya tiene.
    """

    for formato_orig in FORMATOS:

        carpeta = os.path.join(CARPETA_RAIZ, formato_orig)

        if not os.path.isdir(carpeta):
            print(f"⚠ Carpeta no encontrada, se omite: {carpeta}")
            continue

        imagenes = [
            f for f in os.listdir(carpeta)
            if f.lower().endswith(f".{formato_orig}")
        ]

        if not imagenes:
            print(f"⚠ Sin imágenes en: {carpeta}")
            continue

        print(f"\n{'='*55}")
        print(f"📁 Carpeta: {carpeta}  ({len(imagenes)} imágenes)")
        print(f"{'='*55}")

        formatos_salida = [f for f in FORMATOS if f != formato_orig]

        for nombre_archivo in imagenes:

            img_path = os.path.join(carpeta, nombre_archivo)

            print(f"\n  🖼 {nombre_archivo}")

            for metodo in METODOS:
                for formato_sal in formatos_salida:
                    for calidad in CALIDADES:
                        try:
                            df = procesar_imagen(
                                img_path  = img_path,
                                df        = df,
                                metodo    = metodo,
                                formato   = formato_sal,
                                calidad   = calidad
                            )
                        except Exception as e:
                            print(f"    ❌ {metodo:12s} → {formato_sal} q={calidad:2d}  {e}")
                            continue

        print(f"\n✅ Carpeta {formato_orig} completada — filas acumuladas: {len(df)}")

    return df

In [27]:
df_prueba3=pd.DataFrame()
df_prueba3 = procesar_dataset_completo(df_prueba3)


📁 Carpeta: imagenes\jpg  (3 imágenes)

  🖼 COCO_test2014_000000002769.jpg

  🖼 COCO_test2014_000000003905.jpg

  🖼 COCO_test2014_000000004520.jpg

✅ Carpeta jpg completada — filas acumuladas: 90

📁 Carpeta: imagenes\webp  (3 imágenes)

  🖼 t10f_20260105013003_000034_204892.webp

  🖼 t10f_20260105081736_000002_445320.webp

  🖼 t10f_20260105081736_000005_531146.webp

✅ Carpeta webp completada — filas acumuladas: 180

📁 Carpeta: imagenes\png  (3 imágenes)

  🖼 00000000_224resized (2).png

  🖼 5aeca7ba2.png

  🖼 644 (2).png

✅ Carpeta png completada — filas acumuladas: 270


In [25]:
df = pd.DataFrame()
df = procesar_imagen("cute.png", df, "ffmpeg", "jpg",10)

In [19]:
df.dtypes

img_original         object
img_recodificada     object
metodo               object
formato              object
calidad               int64
                     ...   
comp_hist_kurt      float64
ms_ssim             float64
lpips_raw           float64
lpips_norm          float64
combinado           float64
Length: 63, dtype: object

In [28]:
df_prueba3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 63 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   img_original        270 non-null    object 
 1   img_recodificada    270 non-null    object 
 2   metodo              270 non-null    object 
 3   formato             270 non-null    object 
 4   calidad             270 non-null    int64  
 5   orig_rango          270 non-null    float64
 6   orig_std            270 non-null    float64
 7   orig_michelson      270 non-null    float64
 8   orig_hf             270 non-null    float32
 9   orig_lf             270 non-null    float32
 10  orig_hlfr           270 non-null    float32
 11  orig_edges_s1       270 non-null    float64
 12  orig_edges_s2       270 non-null    float64
 13  orig_edges_s3       270 non-null    float64
 14  orig_grad_mean      270 non-null    float32
 15  orig_grad_std       270 non-null    float32
 16  orig_glc